# 106 — Compresión de contexto y cachés semánticos

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** a) HIT (0.94 ≥ 0.90), correcto: misma intención. b) HIT (0.91 ≥ 0.90),
**peligroso**: pregunta por *festivos*, un caso que la respuesta cacheada probablemente
no cubre — falso acierto semántico que el umbral no detecta porque la superficie es casi
idéntica. c) MISS (0.62). Moraleja: los falsos aciertos viven justo encima de τ.

**Ejercicio 2.** Ahorro bruto = `50 000 · 0.35 · 0.006 = $105/día`. Coste de lookup =
`50 000 · 0.0002 = $10/día` (se paga en todas las consultas). Neto ≈ **$95/día**.
Equilibrio: `h · 300 = 10` → `h ≈ 3.3 %`. Casi cualquier hit-rate real es rentable en
dinero; el límite práctico lo ponen los falsos aciertos, no el coste.

**Ejercicio 3.** `[instrucciones del sistema] → [definiciones de herramientas] →
[documentos top-k] → [fecha y hora] → [pregunta del usuario]`. Los dos primeros son
idénticos entre todas las llamadas (prefijo cacheable máximo); los documentos cambian
por consulta pero pueden repetirse entre turnos del mismo hilo; fecha y pregunta son lo
más volátil y van al final para no invalidar nada.

**Ejercicio 4.** El contrato se verifica en el código: `kind == "retrieval"` y
`evidence` no vacía.

In [ ]:
result = run_lab("retrieval", seed=106)
assert result["kind"] == "retrieval"
assert result["evidence"]
show(result)


In [ ]:
tau = 0.90
consultas = [("a", 0.94), ("b", 0.91), ("c", 0.62)]
for nombre, sim in consultas:
    print(nombre, "HIT" if sim >= tau else "MISS", f"(sim={sim})")
print("HIT peligroso: b — 'en festivos' cambia la intención con superficie casi igual")

n, costo_llm, costo_lookup, hit_rate = 50_000, 0.006, 0.0002, 0.35
ahorro_bruto = n * hit_rate * costo_llm
coste_lookup_total = n * costo_lookup
print("ahorro neto diario: $", round(ahorro_bruto - coste_lookup_total, 2))
print("hit-rate de equilibrio:", round(coste_lookup_total / (n * costo_llm), 4))

orden = ["instrucciones del sistema", "definiciones de herramientas",
         "documentos top-k", "fecha y hora", "pregunta del usuario"]
print(" → ".join(orden))

## Reflexión

1. Prompt caching y caché semántico ahorran cosas distintas. ¿Cuál elimina la llamada al LLM y cuál solo abarata el prefill, y por qué pueden (y suelen) convivir?
2. Si subes τ de 0.92 a 0.97, ¿qué pasa con el hit-rate, con los falsos aciertos y con el ahorro neto? ¿Qué datos necesitas para elegir τ de forma defendible?
3. ¿Por qué colocar la fecha actual en la primera línea del system prompt es un error caro con prompt caching, y dónde la colocarías?